##  Gold staging 


In [1]:

# imports only
import os
import requests

import pandas as pd

from dotenv import load_dotenv; load_dotenv()
from databricks.connect import DatabricksSession
import plotly.graph_objects as go
import requests
import plotly.io as pio


In [2]:
t = pio.templates["seaborn"]
# Swap-and-test palettes. Usage: t.layout.colorway = PALETTES["tol_bright"]
PALETTES = {
    "seaborn_colorblind": ["#0173B2", "#DE8F05", "#029E73", "#D55E00",
        "#CC78BC", "#CA9161", "#FBAFE4", "#949494", "#ECE133", "#56B4E9"],
    "okabe_ito": ["#E69F00", "#56B4E9", "#009E73", "#0072B2",
        "#D55E00", "#CC79A7", "#F0E442", "#000000"],
    "tol_bright": ["#88CCEE", "#CC6677", "#DDCC77", "#117733",
        "#332288", "#AA4499", "#44AA99", "#999933",
        "#882255", "#661100", "#888888"],
    "colorbrewer_set2": ["#66C2A5", "#FC8D62", "#8DA0CB", "#E78AC5",
        "#A6D854", "#FFD92F", "#E5C494", "#B3B3B3"],
}
ACTIVE_PALETTE = "okabe_ito"  # change + re-run, then re-run charts
t.layout.colorway = PALETTES[ACTIVE_PALETTE]


In [3]:

t = pio.templates["seaborn"]
pio.templates.default = "seaborn"
t.layout.font.family = "Helvetica, Arial, sans-serif"

t.layout.paper_bgcolor = "white"
t.layout.plot_bgcolor = "white"
HEIGHT, WIDTH = 650, 1000
PAPER = False  # True for journal exports
for ax in (t.layout.xaxis, t.layout.yaxis):
    ax.showgrid = True
    ax.gridcolor = "#cccccc"
    ax.ticks = "outside"
    ax.showline = True
    ax.linecolor = "black"
    if PAPER:
        ax.minor.showgrid = False
        ax.linewidth = 1
    else:
        ax.minor.showgrid = True
        ax.minor.gridcolor = "#eeeeee"
        ax.minor.griddash = "dot"
        ax.mirror = True


In [4]:

spark = DatabricksSession.builder.remote(
    host=os.getenv("DATABRICKS_HOST",""), token=os.getenv("DATABRICKS_TOKEN","")
).serverless(True).getOrCreate()


### Table 1: `gold_daily_fee_metrics` (EXISTS — daily grain, dbt)
- Cadence: 1x/day ~8am on SQL warehouse (`dbt build`), 2-day re-merge window.
- Status: STALE since 2026-10-22 — rebuild pending.

> **sat/vbyte in one line:** a satoshi is the smallest bitcoin unit
  (0.00000001 BTC). Every transaction pays a fee, and miners pick the
  transactions that pay the most *per byte of block space*. So sat/vbyte
  is the going rate for chain space — when blocks fill up, the rate rises
  and cheap transactions wait. p50 = median payer, p90/p99 = what the
  impatient paid.


In [41]:
gold_fee_metrics_q="""select date, p50_sat_vbyte, p90_sat_vbyte, p99_sat_vbyte, tx_count,
       avg_block_fullness_pct, block_count
from workspace.btc_data.gold_daily_fee_metrics
where date < current_date -- exclude partial today
order by date"""

In [42]:


gold_daily = spark.sql(gold_fee_metrics_q).toPandas()
gold_daily.sort_values(by="date", ascending=False, inplace=True)
gold_daily



,date,p50_sat_vbyte,p90_sat_vbyte,p99_sat_vbyte,tx_count,avg_block_fullness_pct,block_count
75,2026-10-30,1.201389,5.000000,15.000000,485180,98.3855954861111,144
74,2026-10-29,0.125984,3.965278,11.540041,578806,97.3572910583942,137
73,2026-10-28,0.121429,3.372973,10.000000,642108,99.3203218750000,144
72,2026-10-27,0.122302,2.881226,8.801802,753710,96.3184115506329,158
71,2026-10-26,0.128000,3.000000,9.181818,822217,98.0589050925926,162
...,...,...,...,...,...,...,...
4,2026-08-12,0.435714,4.800000,16.248619,650968,98.7880127659574,141
3,2026-08-11,0.435714,4.000000,15.507143,608433,98.3927924418605,129
2,2026-08-10,0.410072,4.000000,15.000000,639226,97.3697814748201,139
1,2026-08-09,0.366906,3.000000,12.985714,731544,99.1812503496503,143


### Graph 1-1

In [43]:
HEIGHT, WIDTH = 650, 1000

btc_usd = float(requests.get(
    "https://api.coinbase.com/v2/prices/BTC-USD/spot", timeout=10
).json()["data"]["amount"])
usd_per_sat = btc_usd / 1e8
print(f"1 sat = ${usd_per_sat:.5f} @ ${btc_usd:,.0f}/BTC")

HEIGHT = 400
WIDTH = 700

fig = go.Figure()
for col in ["p50_sat_vbyte", "p90_sat_vbyte", "p99_sat_vbyte"]:
    fig.add_trace(go.Scatter(x=gold_daily["date"], y=gold_daily[col], name=col))
fig.update_layout(showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="date", yaxis_title="sat/vbyte", hovermode="x unified",
    xaxis=dict(showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=14, label="14d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])),
    yaxis=dict(showline=True, linecolor="black", mirror=True))
fig.show()

1 sat = $0.00077 @ $77,273/BTC


### Graph 1-2

In [44]:
g = gold_daily.copy()
g["prem_90"] = g["p90_sat_vbyte"] - g["p50_sat_vbyte"]
g["prem_99"] = g["p99_sat_vbyte"] - g["p50_sat_vbyte"]

fig = go.Figure()
for col in ["prem_90", "prem_99"]:
    fig.add_trace(go.Scatter(x=g["date"], y=g[col], name=col))
fig.update_layout(
    showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="date", yaxis_title="sat/vbyte premium over p50",
    hovermode="x unified",
    xaxis=dict(
        showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=14, label="14d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])),
    yaxis=dict(showline=True, linecolor="black", mirror=True))
fig.show()


---------------------------------------------

### Table 2: whale activity (PROTOTYPE — will become `gold_daily_whale_activity`)
- Source: `workspace.btc_data.transactions` direct (no silver ref).
- Thresholds: shark > 10 BTC, whale > 100 BTC. `output_value` is gross (incl. change).
- BTC-native only; USD happens at dashboard time (`big_btc x close`).
- Query excludes partial today (same rule as Table 1).


In [45]:
whale_q="""select date,
       count_if(output_value > 100) as whale_txs,
       count_if(output_value > 10) as shark_txs,
       sum(if(output_value > 10, output_value, 0)) as big_btc,
       sum(case when output_value > 100 and input_count > 1 and output_count <= 2 then 1 else 0 end) as consolidation,
       sum(case when output_value > 100 and output_count > 2 then 1 else 0 end) as dispersal,
       count(*) as total_txs,
       max(output_value) as top_tx_btc,
       sum(if(output_value > 100, virtual_size, 0)) as whale_vbytes,
       sum(virtual_size) as total_vbytes
from workspace.btc_data.transactions
where is_coinbase = false and date >= current_date - 90
  and date < current_date -- exclude partial today
group by 1 order by 1"""

In [46]:


whale_daily = spark.sql(whale_q).toPandas()
px = spark.sql("""select date, last(close) as btc_usd
from workspace.btc_data.price_minute
where date < current_date group by 1 order by 1""").toPandas()
whale_daily["date"] = pd.to_datetime(whale_daily["date"]).dt.date
px["date"] = pd.to_datetime(px["date"]).dt.date
whale_daily = pd.merge(whale_daily, px, on="date")
whale_daily["transfer"] = whale_daily["whale_txs"] - whale_daily["consolidation"] - whale_daily["dispersal"]
whale_daily["shark_only"] = whale_daily["shark_txs"] - whale_daily["whale_txs"]
whale_daily["whale_vbyte_share"] = whale_daily["whale_vbytes"] / whale_daily["total_vbytes"]
whale_daily.sort_values(by="date", ascending=False, inplace=True)
whale_daily



,date,whale_txs,shark_txs,big_btc,consolidation,dispersal,total_txs,top_tx_btc,whale_vbytes,total_vbytes,btc_usd,transfer,shark_only,whale_vbyte_share
84,2026-10-30,1936,8250,1.255855e+06,464,640,485180,40102.000285,1153180,141777482,76843.14,832,6314,0.008134
83,2026-10-29,1349,5542,8.782638e+05,313,630,578806,20521.324322,759718,133480425,77780.33,406,4193,0.005692
82,2026-10-28,1617,5281,8.614283e+05,380,848,642108,20523.339928,1066670,143154787,81249.31,389,3664,0.007451
81,2026-10-27,1151,4919,8.243843e+05,306,558,753710,20487.923072,661825,152355091,80799.51,287,3768,0.004344
80,2026-10-26,1140,5477,1.147412e+06,314,467,822217,42345.000350,614361,159032171,80008.79,359,4337,0.003863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,2026-08-11,1320,5109,6.549239e+05,258,807,608433,20271.896148,700711,127184176,64050.89,255,3789,0.005509
3,2026-08-10,1281,5385,6.737269e+05,219,804,639226,20259.444861,715476,135593186,64913.95,258,4104,0.005277
2,2026-08-09,842,3669,4.561657e+05,111,631,731544,20263.113279,349452,142121345,64780.53,100,2827,0.002459
1,2026-08-08,896,3633,6.078607e+05,106,677,653738,48225.000452,386458,140743464,64886.48,113,2737,0.002746


### Graph 2-1: shark/whale tx counts


In [47]:
g2 = whale_daily.copy()
g2["shark_only"] = g2["shark_txs"] - g2["whale_txs"]

fig = go.Figure()
fig.add_trace(go.Bar(x=g2["date"], y=g2["shark_only"], name="shark (10-100 BTC)",
    marker_color="rgba(230,159,0,0.35)", marker_line_color="#E69F00", marker_line_width=1.5))
fig.add_trace(go.Bar(x=g2["date"], y=g2["whale_txs"], name="whale (>100 BTC)",
    marker_color="rgba(86,180,233,0.35)", marker_line_color="#56B4E9", marker_line_width=1.5))
fig.add_trace(go.Scatter(x=g2["date"], y=g2["btc_usd"], name="BTC USD", yaxis="y2",
    line=dict(color="black", width=1.5)))
fig.update_layout(
    barmode="stack",
    showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=60),
    xaxis_title="date", yaxis_title="txs / day",
    hovermode="x unified",
    xaxis=dict(
        showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=14, label="14d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])), 

    yaxis=dict(showline=True, linecolor="black", mirror=True, title="txs / day"),
    yaxis2=dict(overlaying="y", side="right", showgrid=False,
                showline=True, linecolor="black", title="BTC USD"))
fig.show()


### Graph 2-4: whale footprint + price (combo of 2-2 and 2-3)
- Stacked shape bars (left) against BTC USD (right, black).
- Question it answers: does dispersal-heavy flow line up with price moves?


In [48]:
fig = go.Figure()
import random
random.seed(36)  # green/orange/purple, max separation
fam = [h for h in PALETTES[ACTIVE_PALETTE] if h != "#000000"]
pick = random.sample(fam, 3)  # [dispersal, consolidation, transfer]
def hx(h, a=0.55):
    h = h.lstrip("#")
    r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{a})"

for col, name, ci in [("dispersal", "dispersal", 0),
                     ("consolidation", "consolidation", 1),
                     ("transfer", "transfer", 2)]:
    fig.add_trace(go.Bar(x=whale_daily["date"], y=whale_daily[col], name=name,
        marker_color=hx(pick[ci], 0.35 if ci == 2 else 0.55),
        marker_line_color=pick[ci], marker_line_width=1))
fig.add_trace(go.Scatter(x=whale_daily["date"], y=whale_daily["btc_usd"], name="BTC USD", yaxis="y2",
    line=dict(color="black", width=1.5)))
fig.update_layout(
    barmode="stack",
    showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=60),
    xaxis_title="date", hovermode="x unified",
    xaxis=dict(
        showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=14, label="14d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])), 
    yaxis=dict(showline=True, linecolor="black", mirror=True, title="whale txs / day"),
    yaxis2=dict(overlaying="y", side="right", showgrid=False,
                showline=True, linecolor="black", title="BTC USD"))
fig.show()


### Table 3: miner revenue (PROTOTYPE — will become `gold_daily_miner_revenue`)
- Subsidy 3.125 BTC/block (post-Apr-2024 halving; whole window qualifies).
- `fees_btc = sum(fee)` ex-coinbase; `revenue = subsidy + fees`; USD via daily close.
- `fee_share_pct` = fees share of revenue — the security-budget number.


In [31]:
rev_q="""with fees as (
    select date, sum(fee) as fees_btc,
           sum(output_value) as settled_btc
    from workspace.btc_data.transactions
    where is_coinbase = false and date >= current_date - 90 and date < current_date
    group by 1
), blks as (
    select date, count(*) as block_count
    from workspace.btc_data.blocks
    where date >= current_date - 90 and date < current_date
    group by 1
), px as (
    select date, last(close) as btc_usd
    from workspace.btc_data.price_minute
    where date >= current_date - 90 and date < current_date
    group by 1
)
select f.date, f.fees_btc, f.settled_btc, b.block_count,
       b.block_count * 3.125 as subsidy_btc,
       b.block_count * 3.125 + f.fees_btc as revenue_btc,
       (b.block_count * 3.125 + f.fees_btc) * p.btc_usd as revenue_usd,
       100 * f.fees_btc / (b.block_count * 3.125 + f.fees_btc) as fee_share_pct,
       p.btc_usd
from fees f
join blks b on f.date = b.date
join px p on f.date = p.date
order by 1"""

In [32]:


rev = spark.sql(rev_q).toPandas()
rev["date"] = pd.to_datetime(rev["date"]).dt.date
g = spark.sql("""select date, tx_count from workspace.btc_data.gold_daily_fee_metrics""").toPandas()
g["date"] = pd.to_datetime(g["date"]).dt.date
rev = pd.merge(rev, g, on="date")
rev["avg_fee_usd"] = rev["fees_btc"] * rev["btc_usd"] / rev["tx_count"]
rev["friction_bps"] = 10000 * rev["fees_btc"] / rev["settled_btc"]
rev.sort_values(by="date", ascending=False, inplace=True)
rev



,date,fees_btc,settled_btc,block_count,subsidy_btc,revenue_btc,revenue_usd,fee_share_pct,btc_usd,tx_count,avg_fee_usd,friction_bps
87,2026-11-02,2.868468,1.260343e+06,127,396.875,399.743468,3.063876e+07,0.717577,76646.05,381759,0.575904,0.022759
86,2026-11-01,2.061920,7.789071e+05,147,459.375,461.436920,3.608237e+07,0.446848,78195.67,703326,0.229244,0.026472
85,2026-10-31,2.175546,7.567468e+05,147,459.375,461.550546,3.572730e+07,0.471356,77407.13,730233,0.230615,0.028749
84,2026-10-30,3.836322,1.336738e+06,144,450.000,453.836322,3.487421e+07,0.845310,76843.14,485180,0.607599,0.028699
83,2026-10-29,2.676597,9.406743e+05,137,428.125,430.801597,3.350789e+07,0.621306,77780.33,578806,0.359683,0.028454
...,...,...,...,...,...,...,...,...,...,...,...,...
4,2026-08-11,2.843089,7.163945e+05,129,403.125,405.968089,2.600262e+07,0.700323,64050.89,608433,0.299297,0.039686
3,2026-08-10,3.045208,7.358119e+05,139,434.375,437.420208,2.839467e+07,0.696175,64913.95,639226,0.309244,0.041386
2,2026-08-09,2.202114,5.034774e+05,143,446.875,449.077114,2.909145e+07,0.490364,64780.53,731544,0.195004,0.043738
1,2026-08-08,2.594629,6.578761e+05,143,446.875,449.469629,2.916450e+07,0.577265,64886.48,653738,0.257529,0.039439


### Graph 3-1: the security squeeze — fee share vs what users pay
- Line (right): fee share of miner revenue % — the subsidy-dependence number. Rises = fees covering more security.
- Orange dash: 7-day mean (trend through daily noise).
- Bars (left): avg fee paid per transaction in USD — what congestion costs users.
- Both move; the old stacked absolute chart couldn't. Footer prints current values.


In [33]:
from typing import Literal
r = rev.copy()
r["date"] = pd.to_datetime(r["date"])
r = r.sort_values("date").reset_index(drop=True)
r["share_7d"] = r["fee_share_pct"].rolling(7, min_periods=3).mean()

fig = go.Figure()
fig.add_trace(go.Bar(x=r["date"], y=r["avg_fee_usd"], name="avg fee / tx",
    marker_color="rgba(115,115,115,0.35)", marker_line_color="#737373",
    marker_line_width=0.6,
    hovertemplate="%{x|%b %d}<br>$%{y:.2f} / tx<extra></extra>"))
fig.add_trace(go.Scatter(x=r["date"], y=r["fee_share_pct"], mode="lines+markers",
    name="fee share of revenue", yaxis="y2",
    line=dict(color="#D55E00", width=2), marker=dict(size=4),
    hovertemplate="%{x|%b %d}<br>%{y:.2f}% of miner revenue<extra></extra>"))
fig.add_trace(go.Scatter(x=r["date"], y=r["share_7d"], mode="lines",
    name="7-day mean", yaxis="y2",
    line=dict(color="#E69F00", width=1.5, dash="dash"), hoverinfo="skip"))

last = r.iloc[-1]
fig.add_annotation(x=1, y=0, xref="x domain", yref="y domain", xanchor="right",
    yanchor="bottom", showarrow=False,
    text=(f"fee share {last['fee_share_pct']:.2f}% (7d {last['share_7d']:.2f}%)  |  "
          f"avg fee ${last['avg_fee_usd']:.2f}/tx  |  friction {last['friction_bps']:.2f} bps"),
    font=dict(size=11))

fig.update_layout(showlegend=True, height=500, width=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=60),
    xaxis_title="date", hovermode="x unified",
    xaxis=dict(showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])),
    yaxis=dict(showline=True, linecolor="black", mirror=True,
               title="avg fee USD / tx", tickformat="$,.2f"),
    yaxis2=dict(showline=True, linecolor="black", mirror=True,
                title="fee share of revenue %", overlaying="y", side="right",
                showgrid=False, ticksuffix="%"))

fig.show()


In [36]:
SUPPLY_BTC = 20_100_000  # approx circulating at block ~974k; scales level only

n = rev.copy()
n["date"] = pd.to_datetime(n["date"])
n = n.sort_values("date").reset_index(drop=True)
n["vol_usd"] = n["settled_btc"] * n["btc_usd"]
n["mcap"] = n["btc_usd"] * SUPPLY_BTC
n["nvt_raw"] = n["mcap"] / n["vol_usd"]
n["vol_90d"] = n["vol_usd"].rolling(90, min_periods=30).mean()
n["nvts"] = n["mcap"] / n["vol_90d"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=n["date"], y=n["nvt_raw"], mode="lines",
    name="NVT raw", line=dict(color="rgba(115,115,115,0.4)", width=1),
    hovertemplate="%{x|%b %d}<br>NVT %{y:.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=n["date"], y=n["nvts"], mode="lines+markers",
    name="NVTS (90d)", line=dict(color="#0072B2", width=2), marker=dict(size=4),
    hovertemplate="%{x|%b %d}<br>NVTS %{y:.0f}<extra></extra>"))

last = n.iloc[-1]
fig.add_annotation(x=1, y=0, xref="x domain", yref="y domain", xanchor="right",
    yanchor="bottom", showarrow=False,
    text=f"NVTS {last['nvts']:.0f} (raw {last['nvt_raw']:.0f})",
    font=dict(size=11))

fig.update_layout(showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="date", yaxis_title="NVT", hovermode="x unified",
    xaxis=dict(showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])),
    yaxis=dict(showline=True, linecolor="black", mirror=True))
fig.show()


### Table 4: when to transact (hour x weekday median fee)
- Grain: hour-of-day x day-of-week, ex-coinbase, with week bucket for window switching.
- `med_sat_vb` inline (`fee*1e8/virtual_size`), no silver ref.
- Spark `dayofweek`: 1=Sunday .. 7=Saturday.


In [93]:
tod_q="""select date_trunc('week', to_date(date)) as wk,
       hour(block_timestamp) as hr, dayofweek(to_date(date)) as dow,
       percentile_approx(fee * 1e8 / virtual_size, 0.5) as med_sat_vb,
       count(*) as txs
from workspace.btc_data.transactions
where is_coinbase = false and virtual_size > 0 and fee is not null
  and date >= current_date - 84 and date < current_date -- 12 weekly buckets
group by 1, 2, 3 order by 1, 3, 2"""

In [94]:


tod = spark.sql(tod_q).toPandas()
tod



,wk,hr,dow,med_sat_vb,txs
0,2026-08-03,0,1,2.000000,12153
1,2026-08-03,1,1,0.438849,19309
2,2026-08-03,2,1,0.366906,35678
3,2026-08-03,3,1,0.342857,52898
4,2026-08-03,4,1,0.345324,34125
...,...,...,...,...,...
2007,2026-10-26,19,6,2.336364,15889
2008,2026-10-26,20,6,2.198556,14465
2009,2026-10-26,21,6,0.120000,50133
2010,2026-10-26,22,6,0.121429,30432


### Graph 4-1: cheapest hours heatmap


In [102]:
tod["wk"] = pd.to_datetime(tod["wk"])
maxwk = tod["wk"].max()
days = ["", "Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
wins = {"4w": 28, "8w": 56, "12w": 84}
pivs = {}
for name, span in wins.items():
    s = tod[tod["wk"] >= maxwk - pd.Timedelta(days=span)]
    pivs[name] = s.pivot_table(index="dow", columns="hr", values="med_sat_vb",
                               aggfunc="median").reindex(range(1, 8))
    b = s.loc[s["med_sat_vb"].idxmin()]
    print(f"{name} cheapest: {days[int(b['dow'])]} {int(b['hr']):02d}:00 UTC ({b['med_sat_vb']:.2f} sat/vb)")

fig = go.Figure(go.Heatmap(z=pivs["4w"].values, x=list(range(24)),
    y=[days[d] for d in pivs["4w"].index], colorscale="YlOrRd",
    xgap=1, ygap=1,
    hovertemplate="%{y} %{x}:00 UTC<br>%{z:.2f} sat/vb"))
fig.update_layout(height=HEIGHT, width=WIDTH,
    plot_bgcolor="black",
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="hour (UTC)", yaxis_title="",
    xaxis=dict(showline=True, linecolor="black", mirror=True),
    yaxis=dict(showline=True, linecolor="black", mirror=True),
    updatemenus=[dict(type="buttons", direction="right", x=0, y=1.12,
        showactive=True, buttons=[
            dict(method="restyle", label=n,
                 args=[{"z": [pivs[n].values]}]) for n in ("4w", "8w", "12w")])])

# width height adjust
fig.update_layout(
    autosize=False,
    width=700,
    height=300,
)
fig.show()


4w cheapest: Tue 04:00 UTC (0.12 sat/vb)
8w cheapest: Tue 04:00 UTC (0.12 sat/vb)
12w cheapest: Tue 04:00 UTC (0.12 sat/vb)


### Table 5: block intervals, 2020-2022 (China-ban shock)
- Grain: day. `interval_sec` from `mediantime` lag by block `number`.
- Expect ~600s baseline; Jun-Jul 2021 should blow past 1000s/day-average.


In [55]:
ivl_q="""with b as (
    select date, number,
           unix_timestamp(mediantime) - unix_timestamp(lag(mediantime) over (order by number)) as interval_sec
    from workspace.btc_data.blocks
    where date >= '2020-01-01' and date < '2023-01-01'
)
select date, avg(interval_sec) / 60 as avg_min, count(*) as blocks
from b where interval_sec > 0 and interval_sec < 7200
group by 1 order by 1"""

In [56]:


ivl = spark.sql(ivl_q).toPandas()
ivl["date"] = pd.to_datetime(ivl["date"]).dt.date
slow = ivl.loc[ivl["avg_min"].idxmax()]
print(f"slowest day: {slow['date']} avg {slow['avg_min']:.1f} min/block over {slow['blocks']:.0f} blocks")
ivl



slowest day: 2021-06-27 avg 20.1 min/block over 58 blocks


,date,avg_min,blocks
0,2020-01-01,8.417052,173
1,2020-01-02,10.020922,141
2,2020-01-03,8.816272,169
3,2020-01-04,8.422355,167
4,2020-01-05,8.933742,163
...,...,...,...
1091,2022-12-27,12.335797,115
1092,2022-12-28,8.453571,168
1093,2022-12-29,10.198818,141
1094,2022-12-30,9.291456,158


### Graph 5-1: average block interval (the ban shock)


In [57]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ivl["date"], y=ivl["avg_min"], name="avg min/block",
    line=dict(color="#E69F00", width=1.5)))
fig.add_hline(y=10, line_dash="dash", line_color="black", annotation_text="10-min target")
vmax = float(ivl["avg_min"].max()) * 1.05
fig.add_trace(go.Scatter(x=[pd.Timestamp("2021-05-21")] * 2, y=[0, vmax],
    mode="lines", name="crackdown May-21",
    line=dict(color="black", dash="dot", width=1.5)))
fig.update_layout(showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="date", yaxis_title="avg minutes / block", hovermode="x unified",
    xaxis=dict(showline=True, linecolor="black", mirror=True),
    yaxis=dict(showline=True, linecolor="black", mirror=True))
fig.show()


### Table 6: fee-tier migration (trailing 365d)
- Grain: day. Tiers on inline `fee*1e8/virtual_size`: <1 / 1-5 / 5-20 / 20+ sat/vb.
- Shares, not levels — shows the market maturing, not just spiking.


In [58]:
tier_q="""with f as (
    select date, fee * 1e8 / virtual_size as sat_vb
    from workspace.btc_data.transactions
    where is_coinbase = false and virtual_size > 0 and fee is not null
      and date >= current_date - 365 and date < current_date
)
select date,
       avg(case when sat_vb < 1 then 1.0 else 0.0 end) as t_sub1,
       avg(case when sat_vb >= 1 and sat_vb < 5 then 1.0 else 0.0 end) as t_1_5,
       avg(case when sat_vb >= 5 and sat_vb < 20 then 1.0 else 0.0 end) as t_5_20,
       avg(case when sat_vb >= 20 then 1.0 else 0.0 end) as t_20p,
       count(*) as txs
from f group by 1 order by 1"""

In [59]:


tiers = spark.sql(tier_q).toPandas()
tiers["date"] = pd.to_datetime(tiers["date"]).dt.date
tiers.sort_values(by="date", inplace=True)
tiers



,date,t_sub1,t_1_5,t_5_20,t_20p,txs
0,2026-01-03,0.44180,0.48078,0.06762,0.00980,501117
1,2026-01-04,0.46069,0.48833,0.04062,0.01037,474444
2,2026-01-05,0.29021,0.62204,0.07373,0.01402,409493
3,2026-01-06,0.32322,0.59804,0.06751,0.01123,424765
4,2026-01-07,0.35120,0.57284,0.06455,0.01141,425044
...,...,...,...,...,...,...
296,2026-10-26,0.69717,0.26613,0.03327,0.00344,822217
297,2026-10-27,0.67064,0.29890,0.02732,0.00314,753710
298,2026-10-28,0.60920,0.34657,0.04029,0.00394,642108
299,2026-10-29,0.56727,0.37757,0.05003,0.00514,578806


### Graph 6-1: fee-tier shares (stacked area)


In [60]:
fig = go.Figure()
for col, name in [("t_sub1", "<1"), ("t_1_5", "1-5"), ("t_5_20", "5-20"), ("t_20p", "20+")]:
    fig.add_trace(go.Scatter(x=tiers["date"], y=tiers[col], name=name, stackgroup="one"))
fig.update_layout(showlegend=True, height=HEIGHT, width=WIDTH,
    legend=dict(title="sat/vb", orientation="h", yanchor="bottom", y=1.02,
                xanchor="right", x=1, font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="date", yaxis_title="share of txs", hovermode="x unified",
    xaxis=dict(showline=True, linecolor="black", mirror=True,
        rangeselector=dict(buttons=[
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=14, label="14d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all")])), 
    yaxis=dict(showline=True, linecolor="black", mirror=True, tickformat=".0%"))
fig.show()


# live blocks — landing view

One block-centric view that grows on every poll: each new block adds one point.

### Graph 1: block interval — how long each block took
- One dot per block: minutes since the previous block.
- Colour = interval on a diverging scale (green fast -> vermillion slow), centred on the 10-min target.
- Size = total transactions in the block, so busy blocks read bigger.
- Shaded band = normal 10-90% range for a 10-min exponential (~1-23 min); dots outside are unusual.
- Orange line = 24-block rolling mean. Black ring = latest block; dotted line = live edge.
- Hover shows block number, txs, and the running schedule lag (blocks behind + / ahead -).


In [138]:
import math

pace_q = """with b as (
    select number, timestamp, transaction_count as txs,
           unix_timestamp(timestamp) - unix_timestamp(lag(timestamp) over (order by number)) as ivl_s
    from workspace.btc_data.blocks
    where timestamp >= current_timestamp() - INTERVAL 24 HOURS
)
select number, timestamp, txs, ivl_s
from b
where ivl_s is not null and ivl_s > 0
order by number"""

p = spark.sql(pace_q).toPandas()
p["timestamp"] = pd.to_datetime(p["timestamp"])
p = p.sort_values("number").reset_index(drop=True)
p["ivl_min"] = p["ivl_s"] / 60.0
ROLL = 24
p["roll"] = p["ivl_min"].rolling(ROLL, min_periods=6).mean()

# running schedule lag: + = behind the 10-min pace, - = ahead
n0 = int(p["number"].iloc[0])
elapsed_min = (p["timestamp"] - p["timestamp"].iloc[0]).dt.total_seconds() / 60.0
p["lag_blocks"] = elapsed_min / 10.0 - (p["number"] - n0)

last = p.iloc[-1]

# marker size scaled to total txs
tmin, tmax = float(p["txs"].min()), float(p["txs"].max())
smin, smax = 6.0, 22.0
if tmax > tmin:
    p["msize"] = smin + (smax - smin) * (p["txs"] - tmin) / (tmax - tmin)
else:
    p["msize"] = 10.0

# diverging scale centred on the 10-min target: green (fast) -> vermillion (slow)
DIV = [[0.0, "#009E73"], [0.5, "#F0E442"], [1.0, "#D55E00"]]
cmid = 10.0
cmax = float(max(20.0, p["ivl_min"].quantile(0.99)))

# 10-90% spread of an exponential with a 10-min mean (what block luck normally looks like)
q10 = -10 * math.log(0.90)
q90 = -10 * math.log(0.10)

ymax = max(25.0, float(p["ivl_min"].quantile(0.99)) * 1.15)
n_above = int((p["ivl_min"] > ymax).sum())

fig = go.Figure()
fig.add_hrect(y0=q10, y1=q90, fillcolor="rgba(0,158,115,0.07)",
              line_width=0, layer="below")
fig.add_hline(y=10, line_dash="dash", line_color="black",
              annotation_text="10-min target", annotation_position="bottom right")

x_last = last["timestamp"].to_pydatetime()
fig.add_shape(type="line", xref="x", yref="y domain", x0=x_last, x1=x_last,
              y0=0, y1=1, line=dict(color="black", dash="dot", width=1))
fig.add_annotation(x=x_last, y=1, xref="x", yref="y domain", text="latest",
                   showarrow=False, xanchor="left", yanchor="bottom",
                   font=dict(size=9))

fig.add_trace(go.Scatter(
    x=p["timestamp"], y=p["ivl_min"], mode="markers", name="block",
    marker=dict(size=p["msize"], color=p["ivl_min"], colorscale=DIV,
                cmin=0.0, cmid=cmid, cmax=cmax, showscale=True,
                colorbar=dict(title="min", thickness=10, len=0.55, y=0.72),
                line=dict(color="black", width=0.4)),
    customdata=list(zip(p["number"].astype(int), p["txs"], p["lag_blocks"])),
    hovertemplate=("block %{customdata[0]}<br>%{x}<br>%{y:.1f} min<br>"
                   "%{customdata[1]:,.0f} txs<br>lag %{customdata[2]:+.0f} blocks"
                   "<extra></extra>")))
fig.add_trace(go.Scatter(x=p["timestamp"], y=p["roll"], mode="lines",
    name=f"{ROLL}-block mean", line=dict(color="#E69F00", width=2.2)))
fig.add_trace(go.Scatter(x=[last["timestamp"]], y=[last["ivl_min"]], mode="markers",
    name="latest", marker=dict(symbol="circle-open", size=22, color="black",
                               line=dict(width=2)), hoverinfo="skip",
    showlegend=False))

fig.add_annotation(x=0, y=q90, xref="x domain", yref="y", xanchor="left",
    yanchor="bottom", showarrow=False, text="normal 10-90%",
    font=dict(size=9, color="#009E73"))
fig.add_annotation(x=1, y=0, xref="x domain", yref="y domain", xanchor="right",
    yanchor="bottom", showarrow=False,
    text=(f"median {p['ivl_min'].median():.1f} min  |  "
          f"{100 * (p['ivl_min'] < 10).mean():.0f}% under 10 min  |  "
          f"{ROLL}-block mean {float(last['roll']):.1f} min"),
    font=dict(size=11))
if n_above:
    fig.add_annotation(x=0, y=1, xref="x domain", yref="y domain", xanchor="left",
        yanchor="top", showarrow=False,
        text=f"{n_above} block(s) above {ymax:.0f} min (clipped)",
        font=dict(size=10, color="#D55E00"))

fig.update_layout(height=450, width=WIDTH, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_title="block time", yaxis_title="minutes since previous block",
    yaxis=dict(range=[0, ymax]))
fig.update_xaxes(showline=True, linecolor="black", mirror=True,
    rangeselector=dict(buttons=[
        dict(count=3, label="3h", step="hour", stepmode="backward"),
        dict(count=12, label="12h", step="hour", stepmode="backward"),
        dict(step="all", label="24h")]))
fig.update_yaxes(showline=True, linecolor="black", mirror=True)
fig.show()


### Graph 2: last 10 blocks — value + price
- Bars = BTC settled per block (gross output value); taller = more value moved.
- Black line (right axis) = BTC price at each block.
- Text above each bar = transactions, then inputs -> outputs.
- Dotted line = 10-block average value.
- Hover: block number and BTC settled.


In [9]:
block_flow_q = """with blk as (
    select number, timestamp
    from workspace.btc_data.blocks
    where number > (select max(number) - 40 from workspace.btc_data.blocks)
), tx as (
    select block_number,
           count(*) as tx_count,
           sum(input_count) as inputs,
           sum(output_count) as outputs,
           sum(output_value) as value_btc,
           sum(fee) as fees_btc
    from workspace.btc_data.transactions
    where is_coinbase = false
      and block_number > (select max(number) - 40 from workspace.btc_data.blocks)
    group by block_number
), px as (
    select candle_start_ts as m, close as btc_usd
    from workspace.btc_data.price_minute
    where candle_start_ts >= current_timestamp() - INTERVAL 2 DAYS
)
select blk.number, blk.timestamp, tx.tx_count, tx.inputs, tx.outputs,
       tx.value_btc, tx.fees_btc, px.btc_usd
from blk
join tx on tx.block_number = blk.number
left join px on px.m = date_trunc('minute', blk.timestamp)
order by blk.number desc
limit 10"""

a = spark.sql(block_flow_q).toPandas()
a["timestamp"] = pd.to_datetime(a["timestamp"])
a = a.sort_values("number").reset_index(drop=True)
for col in ("tx_count", "inputs", "outputs", "value_btc", "fees_btc", "btc_usd"):
    a[col] = a[col].astype(float)
a["btc_usd"] = a["btc_usd"].ffill().bfill()
if a["btc_usd"].isna().all():
    a["btc_usd"] = 0.0
a["label"] = a["timestamp"].dt.strftime("%H:%M")

def fmt(v):
    return f"{v / 1000:.1f}k" if v >= 1000 else f"{v:.0f}"

fig = go.Figure()
fig.add_trace(go.Bar(
    x=a["label"], y=a["value_btc"], name="BTC settled",
    marker_color="#F7931A", marker_line_color="black", marker_line_width=0.6,
    customdata=a["number"].astype(int),
    hovertemplate="block %{customdata}<br>%{y:,.0f} BTC settled<extra></extra>",
    text=[f"{fmt(t)} txs<br>{fmt(i)} in → {fmt(o)} out"
          for t, i, o in zip(a["tx_count"], a["inputs"], a["outputs"])],
    textposition="outside", textfont=dict(size=8)))
fig.add_trace(go.Scatter(
    x=a["label"], y=a["btc_usd"], name="BTC USD", yaxis="y2", mode="lines+markers",
    line=dict(color="black", width=2), marker=dict(size=6),
    hovertemplate="$%{y:,.0f}<extra></extra>"))

avg = float(a["value_btc"].mean())
fig.add_hline(y=avg, line=dict(color="gray", dash="dot", width=1),
              annotation_text=f"avg {avg:,.0f} BTC", annotation_position="top left")

fig.update_layout(height=480, width=WIDTH, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=30, b=40, l=70, r=60),
    xaxis_title="block time (UTC)", yaxis_title="BTC settled per block",
    yaxis=dict(range=[0, float(a["value_btc"].max()) * 1.35]),
    yaxis2=dict(title="BTC USD", overlaying="y", side="right", showgrid=False))
fig.update_xaxes(showline=True, linecolor="black", mirror=True)
fig.update_yaxes(showline=True, linecolor="black", mirror=True)
fig.show()


### Graph 3: intraday price tape — what traders watch
- 1m candles + volume + session VWAP (orange dash) + EMA 9/21 (the intraday trend pair).
- Footer: 24h return % and 24h range — the "returns for 1d" read.
- Trader staple kit, in order: candles + volume, VWAP (fair value), EMA 9/21 cross (momentum),
  RSI (next to add), Bollinger/MACD (volatility/momentum confirm). Orderflow/footprint needs order-book data we don't hold.


In [40]:
px24_q = """select candle_start_ts as ts, open, high, low, close, volume
from workspace.btc_data.price_minute
where candle_start_ts >= current_timestamp() - INTERVAL 24 HOURS
order by 1"""

t = spark.sql(px24_q).toPandas()
for col in ("open", "high", "low", "close", "volume"):
    t[col] = t[col].astype(float)
t["ts"] = pd.to_datetime(t["ts"])
t = t.sort_values("ts").reset_index(drop=True)

# 1m x 1440 candles render ~0.5px wide — resample to 5m (288 bars)
t = (t.set_index("ts")
      .resample("5min")
      .agg({"open": "first", "high": "max", "low": "min",
            "close": "last", "volume": "sum"})
      .dropna().reset_index())

# session VWAP on UTC day + intraday EMAs
t["sess"] = t["ts"].dt.date
t["typ"] = (t["high"] + t["low"] + t["close"]) / 3.0
t["cumvol"] = t.groupby("sess")["volume"].cumsum()
t["vwap"] = (t["typ"] * t["volume"]).groupby(t["sess"]).cumsum() / t["cumvol"].replace(0, float("nan"))
t["ema9"] = t["close"].ewm(span=9, adjust=False).mean()
t["ema21"] = t["close"].ewm(span=21, adjust=False).mean()
t["up"] = t["close"] >= t["open"]

ret_24h = (t["close"].iloc[-1] / t["close"].iloc[0] - 1) * 100
rng = t["close"].max() - t["close"].min()

from plotly.subplots import make_subplots
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.72, 0.28], vertical_spacing=0.05)
fig.add_trace(go.Candlestick(
    x=t["ts"], open=t["open"], high=t["high"], low=t["low"], close=t["close"],
    name="BTC USD", increasing_line_color="#26a69a", increasing_fillcolor="#26a69a",
    decreasing_line_color="#ef5350", decreasing_fillcolor="#ef5350",
    whiskerwidth=0.6), row=1, col=1)
fig.add_trace(go.Scatter(x=t["ts"], y=t["vwap"], mode="lines", name="VWAP",
    line=dict(color="#E69F00", dash="dash", width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t["ts"], y=t["ema9"], mode="lines", name="EMA 9",
    line=dict(color="#0072B2", width=1.2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t["ts"], y=t["ema21"], mode="lines", name="EMA 21",
    line=dict(color="#D55E00", width=1.2)), row=1, col=1)
fig.add_trace(go.Bar(x=t["ts"], y=t["volume"], name="volume", showlegend=False,
    marker_color=["rgba(200,200,200,0.6)" if u else "black" for u in t["up"]],
    marker_line_width=0), row=2, col=1)

fig.add_annotation(x=1, y=0, xref="x domain", yref="y domain", xanchor="right",
    yanchor="bottom", showarrow=False,
    text=f"24h return {ret_24h:+.2f}%  |  range ${rng:,.0f}  |  last ${t['close'].iloc[-1]:,.0f}",
    font=dict(size=11))

# per-window y ranges so zooming x also rescales y (rangeselector can't do that)
def _win(hours):
    w = t[t["ts"] >= t["ts"].max() - pd.Timedelta(hours=hours)] if hours else t
    lo, hi = float(w["low"].min()), float(w["high"].max())
    pad = (hi - lo) * 0.08 or 1.0
    return ([w["ts"].min(), w["ts"].max()],
            [lo - pad, hi + pad],
            [0, float(w["volume"].max()) * 1.25])

_x1h, _y1h, _v1h = _win(1)
_x6h, _y6h, _v6h = _win(6)
_x24h, _y24h, _v24h = _win(0)

fig.update_layout(height=560, width=WIDTH, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
                font=dict(size=10)),
    margin=dict(t=20, b=40, l=60, r=20),
    xaxis_rangeslider_visible=False)
fig.update_xaxes(showticklabels=False)
fig.update_xaxes(showticklabels=True, row=2, col=1)
fig.update_xaxes(showline=True, linecolor="black", mirror=True)
fig.update_layout(updatemenus=[dict(type="buttons", direction="right", x=0, y=1.12,
    showactive=True, buttons=[
        dict(label="1h", method="relayout",
             args=[{"xaxis.range": _x1h, "xaxis2.range": _x1h,
                    "yaxis.range": _y1h, "yaxis2.range": _v1h}]),
        dict(label="6h", method="relayout",
             args=[{"xaxis.range": _x6h, "xaxis2.range": _x6h,
                    "yaxis.range": _y6h, "yaxis2.range": _v6h}]),
        dict(label="24h", method="relayout",
             args=[{"xaxis.range": _x24h, "xaxis2.range": _x24h,
                    "yaxis.range": _y24h, "yaxis2.range": _v24h}])])])
fig.update_yaxes(showline=True, linecolor="black", mirror=True)
fig.update_yaxes(title_text="BTC USD", row=1, col=1)
fig.update_yaxes(title_text="vol", row=2, col=1)
fig.show()
